# Hands-on 5: Simpan Data MQTT ke CSV & SQLite

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 3.

Tujuan:
- Menyimpan data MQTT ke CSV (`data/raw/mqtt_room01.csv`).
- Menyimpan data MQTT ke SQLite (`data/raw/iot_sensor.db`).
- Pola multi-storage: satu subscriber menulis ke CSV + SQLite sekaligus.
- (Opsional, butuh server) Menyimpan ke InfluxDB: `scripts/mqtt/save_to_influxdb.py`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /home/sandbox/iot-ml-book


In [2]:
import subprocess
import sys
import time

PYTHON = sys.executable
LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def start_process(args, stdout_file):
    f = open(stdout_file, "w", encoding="utf-8")
    proc = subprocess.Popen(args, stdout=f, stderr=subprocess.STDOUT)
    return proc, f

# Bersihkan artefak dari eksekusi sebelumnya agar verifikasi bersih.
for p in ["data/raw/mqtt_room01.csv", "data/raw/iot_sensor.db",
          "data/raw/mqtt_room01_combined.csv", "data/raw/iot_sensor_combined.db"]:
    Path(p).unlink(missing_ok=True)
print("[INFO] Artefak lama dibersihkan.")

[INFO] Artefak lama dibersihkan.


## 1. Jalankan Broker dan Publisher

In [3]:
broker_proc, broker_f = start_process(
    [PYTHON, "scripts/mqtt/run_broker.py"], LOG_DIR / "broker_nb5.log"
)
time.sleep(3)
# Publisher buku apa adanya (interval 5 detik).
pub = latar("scripts/mqtt/publish_sensor.py", "publisher_nb5.log")
time.sleep(1)
print("[INFO] Broker dan publisher berjalan.")

[INFO] Broker dan publisher berjalan.


## 2. Simpan ke CSV

`scripts/mqtt/save_to_csv.py` menulis setiap pesan sebagai baris CSV.

In [4]:
# Skrip buku dijalankan apa adanya (loop_forever); dihentikan setelah 18 detik (3 pesan).
print(jalankan_buku("scripts/mqtt/save_to_csv.py", 18))

Terhubung ke broker.
Data disimpan ke CSV: ['room01', 28.76, 47.95, 787, 116.84, '2026-10-22T13:21:27']
Data disimpan ke CSV: ['room01', 25.35, 46.04, 792, 105.07, '2026-10-22T13:21:32']
Data disimpan ke CSV: ['room01', 26.34, 49.47, 599, 144.64, '2026-10-22T13:21:37']



## 3. Simpan ke SQLite

`scripts/mqtt/save_to_sqlite.py` menulis setiap pesan ke tabel `room_telemetry`.

In [5]:
# Skrip buku dijalankan apa adanya (loop_forever); dihentikan setelah 18 detik (3 pesan).
print(jalankan_buku("scripts/mqtt/save_to_sqlite.py", 18))

Terhubung ke broker.
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 26.86, 'humidity_pct': 45.63, 'co2_ppm': 623, 'power_watt': 120.64, 'timestamp': '2026-10-22T13:21:42'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 26.23, 'humidity_pct': 54.89, 'co2_ppm': 615, 'power_watt': 133.77, 'timestamp': '2026-10-22T13:21:47'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 26.81, 'humidity_pct': 56.79, 'co2_ppm': 837, 'power_watt': 141.14, 'timestamp': '2026-10-22T13:21:52'}
Data disimpan ke SQLite: {'device_id': 'room01', 'temperature_c': 27.72, 'humidity_pct': 45.12, 'co2_ppm': 888, 'power_watt': 141.66, 'timestamp': '2026-10-22T13:21:57'}



## 4. Pola Gabungan: CSV + SQLite Sekaligus

`scripts/mqtt/save_to_csv_sqlite.py` menulis ke dua penyimpanan dalam satu alur.

In [6]:
print(jalankan_buku("scripts/mqtt/save_to_csv_sqlite.py", 18))

# Hentikan publisher dan broker.
henti(pub)
broker_proc.terminate(); broker_f.close()
print("[INFO] Publisher dan broker dihentikan.")

Disimpan ke CSV dan SQLite: ['room01', 24.23, 49.53, 535, 118.77, '2026-10-22T13:22:02']
Disimpan ke CSV dan SQLite: ['room01', 25.98, 48.93, 532, 102.47, '2026-10-22T13:22:07']
Disimpan ke CSV dan SQLite: ['room01', 26.93, 50.64, 733, 137.87, '2026-10-22T13:22:12']

[INFO] Publisher dan broker dihentikan.


## 5. Verifikasi Isi Penyimpanan

In [7]:
import pandas as pd
import sqlite3

df_csv = pd.read_csv("data/raw/mqtt_room01.csv")
print("[INFO] CSV mqtt_room01.csv:", df_csv.shape)
display(df_csv.head())

df_comb = pd.read_csv("data/raw/mqtt_room01_combined.csv")
print("[INFO] CSV mqtt_room01_combined.csv:", df_comb.shape)
display(df_comb.head())

[INFO] CSV mqtt_room01.csv: (3, 6)


,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,room01,28.76,47.95,787,116.84,2026-10-22T13:21:27
1,room01,25.35,46.04,792,105.07,2026-10-22T13:21:32
2,room01,26.34,49.47,599,144.64,2026-10-22T13:21:37


[INFO] CSV mqtt_room01_combined.csv: (3, 6)


,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,room01,24.23,49.53,535,118.77,2026-10-22T13:22:02
1,room01,25.98,48.93,532,102.47,2026-10-22T13:22:07
2,room01,26.93,50.64,733,137.87,2026-10-22T13:22:12


In [8]:
conn = sqlite3.connect("data/raw/iot_sensor.db")
df_db = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor.db:", df_db.shape)
display(df_db.head())
conn.close()

conn = sqlite3.connect("data/raw/iot_sensor_combined.db")
df_dbc = pd.read_sql_query("SELECT * FROM room_telemetry", conn)
print("[INFO] SQLite iot_sensor_combined.db:", df_dbc.shape)
display(df_dbc.head())
conn.close()

[INFO] SQLite iot_sensor.db: (4, 7)


,id,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,1,room01,26.86,45.63,623,120.64,2026-10-22T13:21:42
1,2,room01,26.23,54.89,615,133.77,2026-10-22T13:21:47
2,3,room01,26.81,56.79,837,141.14,2026-10-22T13:21:52
3,4,room01,27.72,45.12,888,141.66,2026-10-22T13:21:57


[INFO] SQLite iot_sensor_combined.db: (3, 7)


,id,device_id,temperature_c,humidity_pct,co2_ppm,power_watt,timestamp
0,1,room01,24.23,49.53,535,118.77,2026-10-22T13:22:02
1,2,room01,25.98,48.93,532,102.47,2026-10-22T13:22:07
2,3,room01,26.93,50.64,733,137.87,2026-10-22T13:22:12


## Penutup

Data telemetri MQTT kini tersimpan ke dua bentuk penyimpanan lokal:
CSV (mudah dibaca pandas) dan SQLite (query relasional). Untuk kebutuhan
time-series skala besar, buku menyediakan jalur InfluxDB
(`scripts/mqtt/save_to_influxdb.py`) yang memerlukan server InfluxDB
(Docker `influxdb:2.7`) dan pustaka `influxdb-client`.
Dataset tersimpan ini menjadi bahan eksplorasi pada Hands-on 6.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 5 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 9. Langkah kerja yang harus dilakukan pembaca

Tahap 1 — Jalankan broker
Pastikan Mosquitto aktif.
Tahap 2 — Jalankan publisher
Biarkan simulator sensor mengirim data terus-menerus.
Tahap 3 — Pilih target penyimpanan
Mulai dari CSV, lalu naik ke SQLite, lalu InfluxDB.
Tahap 4 — Jalankan subscriber penyimpanan
Pilih skrip sesuai target.
Tahap 5 — Verifikasi hasil
- buka CSV
- query SQLite
- cek bucket InfluxDB
Tahap 6 — Bandingkan kelebihan
Rasakan perbedaan kenyamanan masing-masing pendekatan.

### 10. Tantangan latihan lanjutan

Agar lebih menantang dan inspiratif, pembaca bisa mencoba:

menyimpan data dari beberapa topic ke satu tabel
menambahkan kolom location
membuat tabel SQLite per jenis sensor
memisahkan bucket InfluxDB untuk room, energy, machine
menambahkan data kualitas udara lain seperti PM2.5
menambahkan alert topic jika suhu > 30°C
membuat skrip pembersihan data mentah menjadi data/processed/

### 11. Insight praktis untuk mahasiswa dan praktisi

Dari tiga media penyimpanan ini, pembaca akan belajar satu hal penting: arsitektur
data harus proporsional dengan kebutuhan.
- Untuk belajar cepat, CSV sudah cukup.
- Untuk aplikasi lokal yang rapi, SQLite sangat berguna.
- Untuk dashboard dan time-series analytics, InfluxDB jauh lebih nyaman.
Tidak semua proyek harus langsung “besar”. Justru pendekatan bertahap seperti ini
lebih cocok untuk laboratorium kampus, bengkel teknik, komunitas maker, dan
implementasi awal di lapangan.